# Занятие 26. Практика: линейная регрессия и цена квартиры

Это второе занятие по теме линейной регрессии: практическая работа в формате учебного рейтинга решений. Команды по 1–2 человека строят модель, сохраняют `submission.csv`, а преподаватель считает результат на скрытой test-выборке.

В блокноте уже есть рабочий стартовый baseline. Сначала запустите его и убедитесь, что получается `submission.csv`, затем улучшайте признаки, модель и validation-качество.

**Задача:** предсказать `price_mln` — цену квартиры в миллионах рублей.

**Главная метрика занятия:** MAE, то есть средняя абсолютная ошибка в млн рублей. Меньше — лучше.

Файл с ответами для test специально не дан в этом блокноте. Для честной проверки не открывайте `competition/data/private_target.csv`, если он лежит рядом в учительской версии материалов.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|---|---:|
| 1 | Загрузка и первичный обзор данных | 4 |
| 2 | Validation и preprocessing | 6 |
| 3 | Первый baseline | 5 |
| 4 | Идеи для улучшения и вывод | 8 |
| 5 | Финальное обучение и submission | 7 |

## Правила и ориентир на 90 минут

1. 0–10 мин — понять данные, метрику и baseline.
2. 10–30 мин — запустить стартовый baseline и получить первый `submission.csv`.
3. 30–70 мин — улучшать признаки и модель на validation.
4. 70–85 мин — выбрать финальную версию и сохранить submission.
5. 85–90 мин — обсудить, какие признаки помогли и почему.

Можно использовать только `train.csv` для обучения и выбора идей. `test.csv` нужен только для финального прогноза.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("competition") / "data"


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(train.shape, test.shape)
display(train.head())
display(sample_submission.head())


## Что есть в данных — **4 балла**

Одна строка — одна квартира из объявления.

Цель `price_mln` есть только в `train.csv`. В `test.csv` её нет: её нужно предсказать.

Подумайте перед моделированием:

- какие признаки числовые по смыслу;
- какие признаки категориальные;
- какие столбцы записаны числами, но по смыслу являются флагами, кодами или порядковыми категориями, а не обычными величинами для расстояния;
- что делать с пропусками в `metro_min`.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — загружены и показаны train.csv и test.csv.
- **1.0 балл** — выполнен первичный обзор данных.
- **1.0 балл** — определены целевой столбец price_mln, id и признаки.
- **1.0 балл** — отмечены числовые, категориальные и проблемные признаки, включая пропуски.

### Снижение баллов
- Цель или id попали в признаки модели → минус **1.0**.
- Нет первичного обзора данных → минус **0.5**.

In [ ]:
train.info()


In [ ]:
train.describe(include="all").T


## Validation: честная проверка своих идей — **6 баллов**

Мы отделяем часть `train.csv` и делаем вид, что ответов к ней нет. Так можно сравнивать идеи до финальной отправки.

### Подробные критерии (для проверки LLM)

- **1.5 балла** — train разделён на train/validation с фиксированным random_state.
- **1.5 балла** — выделены X_train, X_val, y_train, y_val.
- **1.5 балла** — числовые и категориальные признаки обработаны через ColumnTransformer или pipeline.
- **1.5 балла** — преобразования обучаются на train и применяются к validation без утечки.

### Снижение баллов
- Validation не выделена явно → минус **1.0**.
- Preprocessing обучается на полной таблице до split → минус **1.0**.

In [ ]:
target = "price_mln"
id_col = "id"

features = [c for c in train.columns if c not in [target, id_col]]
X = train[features]
y = train[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

print("train:", X_train.shape, "validation:", X_val.shape)


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Числовые:", numeric_features)
print("Категориальные:", categorical_features)


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


numeric_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_ohe()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)


## Первый baseline — **5 баллов**

Baseline — это первая простая модель. Её задача не победить, а дать точку отсчёта: стало ли лучше после ваших идей.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — собран baseline-pipeline.
- **1.0 балл** — baseline обучен на train.
- **1.0 балл** — посчитана MAE на validation.
- **1.0 балл** — посчитана дополнительная метрика или выведены прогнозы для контроля.
- **1.0 балл** — результат baseline сохранён как точка отсчёта.

### Снижение баллов
- Метрика посчитана на train вместо validation → минус **1.0**.
- Baseline не сохранён для дальнейшего сравнения → минус **0.5**.

In [ ]:
baseline_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression()),
    ]
)

baseline_model.fit(X_train, y_train)
val_pred = baseline_model.predict(X_val)

mae = mean_absolute_error(y_val, val_pred)
rmse = np.sqrt(mean_squared_error(y_val, val_pred))
r2 = r2_score(y_val, val_pred)

print(f"MAE:  {mae:.3f} млн руб.")
print(f"RMSE: {rmse:.3f} млн руб.")
print(f"R²:   {r2:.3f}")


## Идеи для улучшения — **8 баллов**

Попробуйте 2–4 идеи и сравнивайте их только по validation:

- добавить признаки `area_per_room`, `kitchen_share`, `is_first_floor`, `is_last_floor`;
- заменить `views_30d` на `log_views = log1p(views_30d)`;
- сравнить `LinearRegression`, `Ridge`, `Lasso`;
- проверить удаление отдельного признака: убрать его, переобучить модель и сравнить MAE на той же validation-выборке;
- проверить, не стала ли модель лучше на MAE, но хуже на RMSE.

Важно: если вы создаёте признак по train, такой же признак нужно создать и в test.

### Подробные критерии (для проверки LLM)

- **2.0 балла** — создано минимум два осмысленных новых признака.
- **2.0 балла** — сравниваются минимум две модели или настройки регуляризации.
- **2.0 балла** — все идеи сравниваются на одной validation-выборке по MAE.
- **1.0 балл** — проверено, не ухудшилась ли RMSE при улучшении MAE.
- **1.0 балл** — есть текстовый вывод о том, что помогло и почему.

### Снижение баллов
- Идеи выбираются по test или скрытым ответам → минус **2.0**.
- Новые признаки созданы для train, но не созданы для test → минус **1.0**.

In [ ]:
def add_features(df):
    df = df.copy()
    df["area_per_room"] = df["area_m2"] / df["rooms"]
    df["kitchen_share"] = df["kitchen_m2"] / df["area_m2"]
    df["is_first_floor"] = (df["floor"] == 1).astype(int)
    df["is_last_floor"] = (df["floor"] == df["floors_total"]).astype(int)
    df["log_views"] = np.log1p(df["views_30d"])
    return df


train_fe = add_features(train)
test_fe = add_features(test)

features_fe = [c for c in train_fe.columns if c not in [target, id_col]]
X = train_fe[features_fe]
y = train_fe[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocess_fe = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess_fe),
        ("model", Ridge(alpha=1.0)),
    ]
)

model.fit(X_train, y_train)
val_pred = model.predict(X_val)

print(f"MAE:  {mean_absolute_error(y_val, val_pred):.3f} млн руб.")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, val_pred)):.3f} млн руб.")
print(f"R²:   {r2_score(y_val, val_pred):.3f}")


**Вывод:**

Baseline нужен как точка отсчёта: без него непонятно, дали ли новые признаки и Ridge реальное улучшение. Дальше идеи сравниваются только на validation, а test используется один раз в финальном файле `submission.csv`.


## Финальное обучение и submission — **7 баллов**

Когда выбрали признаки и модель, обучите её на всём `train.csv`, затем сделайте прогноз для `test.csv`.

### Подробные критерии (для проверки LLM)

- **2.0 балла** — финальная модель обучена на всём train.csv.
- **2.0 балла** — к test.csv применены те же признаки и preprocessing.
- **2.0 балла** — создан submission.csv с колонками id и price_mln.
- **1.0 балл** — количество строк в submission совпадает с test.

### Снижение баллов
- В submission нет id или price_mln → минус **1.0** за каждый столбец.
- Финальный прогноз сделан моделью, не соответствующей выбранной validation-версии → минус **1.0**.

In [ ]:
final_X = train_fe[features_fe]
final_y = train_fe[target]

final_model = model
final_model.fit(final_X, final_y)

test_pred = final_model.predict(test_fe[features_fe])
test_pred = np.maximum(test_pred, 0)  # цена не может быть отрицательной

submission = pd.DataFrame(
    {
        "id": test_fe["id"],
        "price_mln": np.round(test_pred, 3),
    }
)

submission.to_csv("submission.csv", index=False)
submission.head()


## Что сдать

Сдайте файл `submission.csv`.

В нём должны быть ровно два столбца:

- `id`;
- `price_mln`.

Если работаете в команде, переименуйте файл в понятное имя, например `team_ivan_masha.csv`.
